In [ ]:
# PARAMETERS — edit this cell only.
from __future__ import annotations

import json
import os
import re
import sys
from pathlib import Path


def locate_bundle() -> Path:
    starts = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for candidate in starts:
        if (candidate / "data" / "example_tiles.csv").is_file():
            return candidate
        nested = candidate / "tasks" / "260731_2_mllm_notebook_bundle"
        if (nested / "data" / "example_tiles.csv").is_file():
            return nested
    return Path.cwd().resolve()


BUNDLE_DIR = locate_bundle()
PROJECT_DIR = next(
    (candidate for candidate in [BUNDLE_DIR, *BUNDLE_DIR.parents]
     if (candidate / "experiments").is_dir() and (candidate / "scripts").is_dir()),
    BUNDLE_DIR,
)
sys.path.append(str(PROJECT_DIR))

# Required for live tile loading. Leave empty for the included six-tile examples.
COUNTING_DATA_ROOT = os.environ.get("COUNTING_DATA_ROOT", "")
TASKS = [
    "malaria/trophozoite", "malaria/difficult", "malaria/red_blood_cell",
    "birds/bird", "dota/tennis_court", "dota/harbor", "dota/ship",
    "dota/plane", "dota/small_vehicle", "kenyan_wildlife/giraffe",
    "dota/helicopter", "dota/storage_tank",
]
MODEL_LABEL = "Qwen3-VL-8B-Instruct"
BATCH_SIZE = 8
MAX_CHOICE = 300
GPU_MEMORY_UTILIZATION = 0.85
LABEL_ROOT = PROJECT_DIR / "outputs" / "mllm_all_tiles_all_models"
PROPRIETARY_ROOT = PROJECT_DIR / "outputs" / "proprietary_all_tiles_260710"
RUN_ROOT = PROJECT_DIR / "outputs" / "runs"
EXAMPLE_CSV = BUNDLE_DIR / "data" / "example_tiles.csv"
OPEN_SOURCE_CSV = PROJECT_DIR / "outputs" / "runs" / "open_source_all_tiles_current.csv" if (PROJECT_DIR / "outputs" / "runs" / "open_source_all_tiles_current.csv").is_file() else BUNDLE_DIR / "data" / "open_source_reference_sample.csv"
PROPRIETARY_CSV = PROJECT_DIR / "outputs" / "runs" / "proprietary_all_tiles_current.csv" if (PROJECT_DIR / "outputs" / "runs" / "proprietary_all_tiles_current.csv").is_file() else BUNDLE_DIR / "data" / "proprietary_reference_sample.csv"
OPENROUTER_KEY_SELECTOR = "OPENROUTER_API_KEY"

try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except NameError:
    pass

OPEN_SOURCE_CACHE_ROOT = PROJECT_DIR / "outputs" / "mllm_all_tiles_all_models"
PROPRIETARY_INTERPRETED_ROOT = PROJECT_DIR / "outputs" / "proprietary_all_tiles_260710"
EXPORT_DIR = BUNDLE_DIR / "exports"
WRITE_EXPORTS = False
OPEN_SOURCE_EXPORT = EXPORT_DIR / "open_source_all_tiles.csv"
PROPRIETARY_EXPORT = EXPORT_DIR / "proprietary_all_tiles.csv"


# 06 — Export compact CSVs

The final exports intentionally contain only the fields consumed by downstream analysis: dataset, task, tile id, truth, model label, and final value. The open-source export reads JSONL label caches; the proprietary export reads interpreted staged records when available and otherwise previews the published CSV. Set `WRITE_EXPORTS=True` to write new files.


In [ ]:
import pandas as pd

FIELDS = ["dataset", "task", "tileid", "ground_truth_label", "model_label", "final_value"]
OPEN_SOURCE_MODELS = ["Qwen3-VL-8B-Instruct", "InternVL3.5-8B", "Molmo-7B-D-0924", "Aya-Vision-8B", "Molmo2-8B"]

def compact_row(class_key: str, model_label: str, row: dict) -> dict:
    dataset, task = class_key.split("/", 1)
    answer = float(row["answer"])
    if not answer.is_integer() or answer < 0:
        raise ValueError(f"Invalid answer for {class_key} tile {row.get('tile_index')}: {row['answer']!r}")
    return {"dataset": dataset, "task": task, "tileid": int(row["tile_index"]), "ground_truth_label": int(row["gt_count"]), "model_label": model_label, "final_value": int(answer)}

def export_open_source() -> pd.DataFrame:
    rows = []
    diagnostics = {"invalid_json": 0, "incomplete": 0, "duplicate_overrides": 0}
    for model_label in OPEN_SOURCE_MODELS:
        for class_key in TASKS:
            dataset, task = class_key.split("/", 1)
            path = OPEN_SOURCE_CACHE_ROOT / model_label / f"{dataset}__{task}.jsonl"
            if not path.is_file():
                continue
            seen = {}
            for line in path.read_text().splitlines():
                if line.strip():
                    try:
                        record = json.loads(line)
                    except json.JSONDecodeError:
                        diagnostics["invalid_json"] += 1
                        continue
                    if "tile_index" not in record or "answer" not in record:
                        diagnostics["incomplete"] += 1
                        continue
                    if record.get("class_key") != class_key or record.get("model_label") != model_label:
                        raise ValueError(f"Unexpected provenance in {path}")
                    if int(record["tile_index"]) in seen:
                        diagnostics["duplicate_overrides"] += 1
                    seen[int(record["tile_index"])] = record
            rows.extend(compact_row(class_key, model_label, seen[index]) for index in sorted(seen))
    print({"open_source_diagnostics": diagnostics})
    return pd.DataFrame(rows, columns=FIELDS)

def export_proprietary() -> pd.DataFrame:
    rows = []
    for path in sorted(PROPRIETARY_INTERPRETED_ROOT.glob("07_interpreted_outputs/*/*.json")):
        record = json.loads(path.read_text())
        rows.append(compact_row(record["class_key"], record["model_label"], {"tile_index": record["tile_index"], "gt_count": record["tile"]["gt_count"], "answer": record["answer"]}))
    if rows:
        return pd.DataFrame(rows, columns=FIELDS)
    return pd.read_csv(PROPRIETARY_CSV, usecols=FIELDS) if PROPRIETARY_CSV.is_file() else pd.DataFrame(columns=FIELDS)

open_source_export = export_open_source()
proprietary_export = export_proprietary()
if WRITE_EXPORTS:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    open_source_export.to_csv(OPEN_SOURCE_EXPORT, index=False)
    proprietary_export.to_csv(PROPRIETARY_EXPORT, index=False)
print({"open_source_rows": len(open_source_export), "proprietary_rows": len(proprietary_export), "write_exports": WRITE_EXPORTS})


## Visualize exported error distributions

The export is only useful if the row key is preserved. This plot recomputes absolute error from the compact table and compares the model families on the same scale.


In [ ]:
from mu.visv2.figure import Figure

frames = []
for frame, family in [(open_source_export, "open-source"), (proprietary_export, "proprietary")]:
    if not frame.empty:
        copy = frame.copy()
        copy["absolute_error"] = (copy["final_value"] - copy["ground_truth_label"]).abs()
        copy["family"] = family
        frames.append(copy)
all_exports = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
if all_exports.empty:
    print("No export rows available for visualization.")
else:
    fig = Figure(figsize=(9, 4.5), fontfamily="sans-serif", fontsize=11)
    panel = fig.panels[0]
    for family, frame in all_exports.groupby("family"):
        panel.p.hist(frame["absolute_error"], bins=35, alpha=0.48, label=family)
    panel.set_xaxislabel("absolute error")
    panel.set_yaxislabel("rows")
    panel.set_title("Compact export: error distribution")
    panel.p.legend(frameon=False)
    fig.show()
